# Day 04 - TV-layout elementwise operations and predication

Day 04 connects the first half of official
[Notebook 08](../notebooks/08_elementwise_add.ipynb) to a reusable CuTe DSL
kernel pattern. Notebook 08 progresses through:

1. one scalar element per thread;
2. a `(1, 8)` vector per thread;
3. CTA tiling plus a Thread-Value (TV) layout;
4. a generic elementwise operator.

Day 03 covers the first two stages. Day 04 starts at stage three, makes the
thread/value ownership explicit, and adds coordinate-based predication so
partial edge tiles are correct. The elementwise operation is still just
`C = A + B`; the challenge is mapping each thread to the right addresses.

## How this challenge fits Notebook 08

Official [Notebook 08](../notebooks/08_elementwise_add.ipynb) builds one
elementwise operation through four representations:

| Stage | Ownership expression | Local lesson |
| --- | --- | --- |
| Scalar | one global thread ID selects one `(m, n)` | Day 03 part 1 |
| Vector | one thread selects a `(1, 8)` slice | Day 03 part 2 |
| TV layout | `(thread_id, value_id)` maps into a CTA tile | **Day 04** |
| Generic operator | the same mapping calls a compile-time operation | Extension after Day 04 |

The arithmetic does not become harder. The representation of ownership becomes
more explicit and reusable.

In [ ]:
import torch

import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

try:
    import cutlass.testing as testing
except ModuleNotFoundError as error:
    if error.name != "cutlass.testing":
        raise
    from cutlass.cute import testing

cutlass.cuda.initialize_cuda_context()

## 1. From thread IDs to TV layouts

The Day 03 vectorized kernel calculates a global thread index manually and
uses it to select one `(1, 8)` vector. That works for one simple mapping, but
the index arithmetic becomes hard to reuse for larger thread tiles.

A TV layout packages ownership as a function:

```text
tv_layout(thread_id, value_id) -> (tile_m, tile_n)
```

`thread_id` identifies a thread in the CTA. `value_id` identifies one value
owned by that thread. The result is a logical coordinate inside one CTA tile.
The data tensor layout then maps that logical coordinate to a memory address.

For FP16 in this lesson:

```python
elts_per_vec = 128 // 16  # 8 FP16 values in 128 bits
thr_layout = cute.make_ordered_layout((4, 32), order=(1, 0))
val_layout = cute.make_ordered_layout((4, 8), order=(1, 0))
tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)
```

There are `4 * 32 = 128` threads. Each thread owns `4 * 8 = 32` values.
Together they cover `128 * 32 = 4096` values, arranged as a
`(4 * 4, 32 * 8) = (16, 256)` CTA tile.

The two returned objects answer different questions:

- `tiler_mn` says **how large one CTA tile is**: `(16, 256)`.
- `tv_layout` says **which `(tid, vid)` owns each coordinate in that tile**.

In [ ]:
# Calculate the expected FP16 tile without compiling a kernel.
dtype_bits = 16
elts_per_vec = 128 // dtype_bits
thread_shape = (4, 32)
value_shape = (4, elts_per_vec)
tile_shape = (
    thread_shape[0] * value_shape[0],
    thread_shape[1] * value_shape[1],
)
threads = thread_shape[0] * thread_shape[1]
values_per_thread = value_shape[0] * value_shape[1]

assert threads == 128
assert values_per_thread == 32
assert tile_shape == (16, 256)
assert threads * values_per_thread == tile_shape[0] * tile_shape[1]
print(
    f"{threads} threads x {values_per_thread} values/thread "
    f"= tile {tile_shape}"
)

## 2. What `zipped_divide` does

For an input tensor with logical shape `(M, N)`:

```python
gA = cute.zipped_divide(mA, tiler_mn)
```

conceptually changes the domain to:

```text
((TileM, TileN), (RestM, RestN))
```

The first mode is a coordinate inside a tile. The second mode chooses a tile.
`RestM` and `RestN` use ceiling division, so the last tile may extend beyond
the original tensor. No data is padded or allocated by `zipped_divide`; it is
a layout transformation.

For `(M, N) = (33, 513)` and tile `(16, 256)`:

```text
RestM = ceil(33 / 16)  = 3
RestN = ceil(513 / 256) = 3
```

The launch therefore needs nine CTAs. CTA tiles on the bottom and right edges
contain logical coordinates outside the real tensor and must be predicated.

In [ ]:
def ceil_div(a, b):
    return (a + b - 1) // b


M, N = 33, 513
TileM, TileN = 16, 256
RestM, RestN = ceil_div(M, TileM), ceil_div(N, TileN)
assert (RestM, RestN) == (3, 3)
print(f"CTA grid has {RestM} x {RestN} = {RestM * RestN} tiles")

## 3. The mapping chain inside the kernel

Each CTA first selects its tile:

```python
blk_coord = ((None, None), bidx)
blkA = gA[blk_coord]
```

`None` preserves the complete inner tile while `bidx` selects one outer tile.
The resulting `blkA` maps:

```text
(tile_m, tile_n) -> global-memory address
```

Composition connects TV ownership to that address mapping:

```python
tidfrgA = cute.composition(blkA, tv_layout)
```

Mathematically:

```text
(tid, vid) --tv_layout--> (tile_m, tile_n) --blkA--> address
```

so `tidfrgA` directly maps `(tid, vid)` to an address. Slicing with
`(tidx, None)` fixes the thread ID and preserves every value owned by it:

```python
thrA = tidfrgA[(tidx, None)]  # vid -> address
```

The same mapping is applied to A, B, C, and the coordinate tensor. Identical
partitioning is what keeps data and bounds checks aligned.

## 4. Why the identity tensor is the predicate

`cute.make_identity_tensor((M, N))` is a coordinate-valued tensor:

```text
idC(m, n) = (m, n)
```

It does not allocate an `(M, N)` data buffer. After the same
`zipped_divide`, CTA selection, composition, and thread selection as C,
`thrCrd[vid]` gives the global `(m, n)` coordinate associated with
`thrC[vid]`.

The predicate is therefore:

```python
frgPred[vid] = cute.elem_less(thrCrd[vid], shape)
```

Both coordinates must be inside the original shape. Apply the predicate to
loads and stores so invalid edge lanes never access memory.

## 5. Puzzle TODO sequence

Implement one ownership transition at a time in `puzzle.py` or `day04.ipynb`:

1. **CTA tile:** slice A, B, C, and coordinates with `blk_coord`.
2. **TV composition:** compose every CTA-local tensor with `tv_layout`.
3. **Thread slice:** fix `tidx` and preserve the value mode.
4. **Predicate:** compare each coordinate against `(M, N)`.
5. **Elementwise body:** predicated load, `A + B`, predicated store.

At each stage, ask what the current tensor maps **from** and **to**. If that
sentence is unclear, print its type before writing the next line.

## Fill the elementwise kernel

The host constructs `tiler_mn`, `tv_layout`, tiled data tensors, and the tiled
identity tensor. Fill the six TODOs in order. Running this cell only defines
the functions; the `NotImplementedError` is reached when the kernel is
compiled in the correctness cell.

In [ ]:
@cute.kernel
def elementwise_add_kernel(
    gA: cute.Tensor,
    gB: cute.Tensor,
    gC: cute.Tensor,
    cC: cute.Tensor,
    shape: cute.Shape,
    tv_layout: cute.Layout,
):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()

    # TODO(1): Select this CTA's data and coordinate tiles.
    #
    # 1. Slice each of gA, gB, gC, cC down to this CTA's tile:
    #        blk_coord = ((None, None), bidx)
    #        blkA = gA[blk_coord]            etc.
    # TODO(2): Compose each block with `tv_layout`.
    #
    # The mapping chain is:
    #   (tid, vid) --tv_layout--> (tile_m, tile_n)
    #              --blkA-------> global-memory address
    #
    # Compose each block so the result maps (tid, vid) -> address:
    #        tidfrgA   = cute.composition(blkA,   tv_layout)
    #        ...
    #        tidfrgCrd = cute.composition(blkCrd, tv_layout)
    #
    # TODO(3): Slice the composed tensors to this thread.
    #
    # Use thr_coord = (tidx, None):
    #        thrA   = tidfrgA[thr_coord]
    #        ...
    #        thrCrd = tidfrgCrd[thr_coord]
    #
    # TODO(4): Reprofile each thread tensor as `(1, num_values)`.
    #
    # `cute.copy` treats mode 0 as its V-mode and predicates the remaining
    # modes. A scalar V-mode therefore preserves one predicate per value:
    #        copy_layout = cute.make_layout(
    #            (1, cute.size(thrCrd)), stride=(0, 1)
    #        )
    #        copyA = cute.composition(thrA, copy_layout)
    #        ...
    #        copyCrd = cute.composition(thrCrd, copy_layout)
    #
    # TODO(5): Build a per-value predicate from the reprofiled coordinates:
    #        frgPred = cute.make_rmem_tensor(copyCrd.shape, cutlass.Boolean)
    #        for i in cutlass.range_constexpr(cute.size(frgPred)):
    #            frgPred[i] = cute.elem_less(copyCrd[i], shape)
    #
    # TODO(6): Create a universal copy atom, use predicated `cute.copy` to load
    # A and B into register fragments, apply the elementwise operation, then
    # use predicated `cute.copy` to store C.
    #
    # The arithmetic is intentionally the easy part. Let CuTe own the
    # per-element copy traversal:
    #        frgA = cute.make_fragment_like(copyA)
    #        copy_atom = cute.make_copy_atom(
    #            cute.nvgpu.CopyUniversalOp(),
    #            gA.element_type,
    #            num_bits_per_copy=gA.element_type.width,
    #        )
    #        cute.copy(copy_atom, copyA, frgA, pred=frgPred)
    #        cute.copy(copy_atom, copyB, frgB, pred=frgPred)
    #        result = frgA.load() + frgB.load()
    #        frgC.store(result)
    #        cute.copy(copy_atom, frgC, copyC, pred=frgPred)
    raise NotImplementedError("Day 04: fill in the TV-layout kernel")


@cute.jit
def elementwise_add(mA: cute.Tensor, mB: cute.Tensor, mC: cute.Tensor):
    assert mA.element_type == mB.element_type == mC.element_type
    dtype = mA.element_type

    elts_per_vec: cutlass.Constexpr = 128 // dtype.width
    thr_layout = cute.make_ordered_layout((4, 32), order=(1, 0))
    val_layout = cute.make_ordered_layout((4, elts_per_vec), order=(1, 0))
    tiler_mn, tv_layout = cute.make_layout_tv(thr_layout, val_layout)
    print(f"[JIT] tiler_mn={tiler_mn} tv_layout={tv_layout}")

    gA = cute.zipped_divide(mA, tiler_mn)
    gB = cute.zipped_divide(mB, tiler_mn)
    gC = cute.zipped_divide(mC, tiler_mn)

    idC = cute.make_identity_tensor(mC.shape)
    cC = cute.zipped_divide(idC, tiler=tiler_mn)

    elementwise_add_kernel(gA, gB, gC, cC, mC.shape, tv_layout).launch(
        grid=(cute.size(gC, mode=[1]), 1, 1),
        block=(cute.size(tv_layout, mode=[0]), 1, 1),
    )

## 6. Replace manual loops with `cute.copy`

A first implementation can copy each owned value explicitly:

```python
for i in cutlass.range_constexpr(cute.size(frgA)):
    if frgPred[i]:
        frgA[i] = thrA[i]
```

That is useful while learning ownership, but the traversal is boilerplate.
Once source, destination, and predicate have compatible profiles, create a copy
atom and let `cute.copy` traverse them:

```python
copy_layout = cute.make_layout((1, cute.size(thrA)), stride=(0, 1))
copyA = cute.composition(thrA, copy_layout)
copyB = cute.composition(thrB, copy_layout)
copyC = cute.composition(thrC, copy_layout)

copy_atom = cute.make_copy_atom(
    cute.nvgpu.CopyUniversalOp(),
    gA.element_type,
    num_bits_per_copy=gA.element_type.width,
)

cute.copy(copy_atom, copyA, frgA, pred=frgPred)
cute.copy(copy_atom, copyB, frgB, pred=frgPred)
frgC.store(frgA.load() + frgB.load())
cute.copy(copy_atom, frgC, copyC, pred=frgPred)
```

`cute.copy` interprets its input profile as `(V, Rest...)`: the copy atom
consumes `V`, while `pred` must match `Rest...`. Reprofiling to
`(1, num_values)` makes `V` scalar and leaves one predicate per owned value,
which is required for arbitrary partial edge tiles. The atom describes the
elementary transfer operation, the layouts describe traversal, and `pred`
suppresses invalid edge accesses. This removes three manual loops without
changing tile ownership or bounds semantics.

## 7. Correctness before performance

Use both a divisible shape and a partial-tile shape:

```bash
python puzzle.py --M 1024 --N 1024
python puzzle.py --M 1023 --N 1025
python solution.py --M 1023 --N 1025
```

The second shape is the important one: it exercises both row and column
predicates. Always synchronize before checking because kernel launches are
asynchronous.

In [ ]:
# Run after filling the six kernel TODOs.
M, N = 1023, 1025
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.zeros_like(a)

a_, b_, c_ = (
    from_dlpack(t, assumed_align=16).mark_layout_dynamic()
    for t in (a, b, c)
)
compiled = cute.compile(elementwise_add, a_, b_, c_)
compiled(a_, b_, c_)
torch.cuda.synchronize()
torch.testing.assert_close(c, a + b)
print("Correct on the partial-tile shape (1023, 1025)")

## 8. Benchmarking

Compile once, then benchmark only the compiled callable:

```python
try:
    import cutlass.testing as testing       # CuTe DSL 4.6+
except ModuleNotFoundError as error:
    if error.name != "cutlass.testing":
        raise
    from cutlass.cute import testing        # older Notebook 08 API

compiled = cute.compile(elementwise_add, a_, b_, c_)
avg_time_us = testing.benchmark(
    compiled,
    kernel_arguments=testing.JitArguments(a_, b_, c_),
    warmup_iterations=5,
    iterations=100,
)
```

Do not include JIT compilation or tensor allocation in kernel timing.
Elementwise FP16 add transfers two inputs and one output:

```text
bytes = M * N * (2 reads + 1 write) * 2 bytes = 6 * M * N
GB/s  = bytes / (time_us * 1000)
```

This is **effective algorithmic bandwidth**. It counts the required tensor
traffic, not every physical transaction or cache effect.

```bash
python solution.py --M 16384 --N 8192 --benchmark
python solution.py --M 16384 --N 8192 --benchmark \
  --warmup 10 --iterations 200
```

Use a large shape for stable timing. Keep the odd shape for correctness, not
for peak-bandwidth comparison.

In [ ]:
# Benchmark only after correctness passes. Use a large, regular shape.
M, N = 16384, 8192
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c = torch.empty_like(a)
a_, b_, c_ = (
    from_dlpack(t, assumed_align=16).mark_layout_dynamic()
    for t in (a, b, c)
)

compiled = cute.compile(elementwise_add, a_, b_, c_)
compiled(a_, b_, c_)
torch.cuda.synchronize()
torch.testing.assert_close(c, a + b)

avg_time_us = testing.benchmark(
    compiled,
    kernel_arguments=testing.JitArguments(a_, b_, c_),
    warmup_iterations=5,
    iterations=100,
)
total_bytes = 3 * a.numel() * a.element_size()
bandwidth_gbps = total_bytes / (avg_time_us * 1_000)
print(f"time: {avg_time_us:.3f} us")
print(f"effective bandwidth: {bandwidth_gbps:.2f} GB/s")

## Extension: from add to a generic elementwise operation

Notebook 08's next step passes an operation as a compile-time argument. The
ownership pipeline stays unchanged:

```text
tile -> compose TV layout -> thread slice -> predicate -> load -> op -> store
```

Only the compute line changes conceptually from:

```python
result = frgA.load() + frgB.load()
```

to:

```python
result = op(frgA.load(), frgB.load())
```

After finishing addition, try multiplication and `relu(a * b)`. Keep the same
coordinate predicate and verify each operator against a PyTorch expression.
This demonstrates why layout code and mathematical operation should be
separate.

## Reference implementation

Use the reference only after completing the six TODOs:

```python
%run solution.py --M 1023 --N 1025
%run solution.py --M 16384 --N 8192 --benchmark
```

## What you should know afterward

- A tiler defines the CTA tile shape; a TV layout defines ownership inside it.
- `zipped_divide` separates intra-tile and inter-tile coordinates.
- Composition turns `(tid, vid)` into an address without manual index math.
- An identity tensor carries coordinates through the exact same partitioning.
- Elementwise kernels are usually memory-bound, so report time and effective
  bandwidth after proving correctness.
- Notebook 08's generic elementwise operator changes step 5; the tiling,
  ownership, and predicate skeleton stay the same.